PROSES 1: Exploratory Data Analysis (EDA) dan Data Preprocessing
Dataset UJIIndoorLoc. Target klasifikasi: BUILDINGID, FLOOR. Target regresi: LONGITUDE, LATITUDE.

Letakkan trainingData.csv dan (opsional) validationData.csv di folder yang sama dengan notebook. Ubah path jika diperlukan. Jalankan cell secara berurutan.

1. Import library dan konfigurasi

In [7]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
TRAIN_PATH = Path("data/trainingData.csv")
VALIDATION_PATH = Path("data/validationData.csv")
TARGET_CLASS = ["BUILDINGID", "FLOOR"]
TARGET_REG = ["LONGITUDE", "LATITUDE"]
TARGETS = TARGET_CLASS + TARGET_REG
METADATA = ["USERID", "PHONEID", "TIMESTAMP"]
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

2. Memuat Dataset

In [9]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError("trainingData.csv tidak ditemukan. Sesuaikan TRAIN_PATH.")
train_raw = pd.read_csv(TRAIN_PATH)
validation_raw = pd.read_csv(VALIDATION_PATH) if VALIDATION_PATH.exists() else None
print("Training:", train_raw.shape)
print("Validation:", None if validation_raw is None else validation_raw.shape)
train_raw.head()

Training: (19937, 529)
Validation: (1111, 529)


,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,WAP011,WAP012,WAP013,WAP014,WAP015,...,WAP515,WAP516,WAP517,WAP518,WAP519,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,-7541.2643,4.864921e+06,2,1,106,2,2,23,1371713733
1,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,-7536.6212,4.864934e+06,2,1,106,2,2,23,1371713691
2,100,100,100,100,100,100,100,-97,100,100,100,100,100,100,100,...,100,100,100,100,100,100,-7519.1524,4.864950e+06,2,1,103,2,2,23,1371714095
3,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,-7524.5704,4.864934e+06,2,1,102,2,2,23,1371713807
4,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100,...,100,100,100,100,100,100,-7632.1436,4.864982e+06,0,0,122,2,11,13,1369909710


3. Struktur data, missing value, dan duplikasi

In [13]:
display(pd.DataFrame({
    "dtype": train_raw.dtypes.astype(str),
    "missing": train_raw.isna().sum(),
    "missing_percent": (train_raw.isna().mean()*100).round(2),
    "unique": train_raw.nunique(dropna=False)
}).sort_values("missing_percent", ascending=False))
print("Duplikasi penuh:", train_raw.duplicated().sum())
print("Kolom:", train_raw.shape[1], "| Baris:", train_raw.shape[0])

,dtype,missing,missing_percent,unique
WAP001,int64,0,0.0,6
WAP002,int64,0,0.0,3
WAP003,int64,0,0.0,1
WAP004,int64,0,0.0,1
WAP005,int64,0,0.0,10
...,...,...,...,...
SPACEID,int64,0,0.0,123
RELATIVEPOSITION,int64,0,0.0,2
USERID,int64,0,0.0,18
PHONEID,int64,0,0.0,16


Duplikasi penuh: 637
Kolom: 529 | Baris: 19937
